In [8]:
import torch
from transformers import Gemma2Model, AutoTokenizer
from textboost.adapters import add_adapter_to_model, TrfConfig

torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    "Efficient-Large-Model/SANA1.5_1.6B_1024px_diffusers",
    subfolder="tokenizer",
)
model = Gemma2Model.from_pretrained(
    "Efficient-Large-Model/SANA1.5_1.6B_1024px_diffusers",
    subfolder="text_encoder",
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.72it/s]


In [12]:
embeddings = model.get_input_embeddings().weight.data.detach().clone()

# Remove 0-norm embeddings.
print(embeddings.shape)
norms = embeddings.norm(dim=-1)
embeddings = embeddings[norms > 0]
print(embeddings.shape)

min_norm = norms.min().item()
print(f"min norm: {min_norm}")
mean_norm = norms.mean().item()
print(f"mean norm: {mean_norm}")
max_norm = norms.max().item()
print(f"max norm: {max_norm}")

# Sort by norm.
sorted_norms, sorted_indices = torch.sort(norms, descending=True)
print(sorted_norms)
print(sorted_indices)
for i in range(20):
    idx = sorted_indices[i].item()
    print(f"idx: {idx}, norm: {sorted_norms[i].item()}")
    print(tokenizer.decode([idx]))

torch.Size([256000, 2304])
torch.Size([256000, 2304])
min norm: 1.0645419359207153
mean norm: 1.7720332145690918
max norm: 4.863350868225098
tensor([4.8634, 4.7131, 4.6965,  ..., 1.0717, 1.0654, 1.0645])
tensor([234323, 213525, 212562,  ...,  32436, 246077,  62024])
idx: 234323, norm: 4.863350868225098
 myſelf
idx: 213525, norm: 4.713084697723389
 itſelf
idx: 212562, norm: 4.696544647216797
 Efq
idx: 234446, norm: 4.5944294929504395
 Monfieur
idx: 220758, norm: 4.501970291137695
 Theſe
idx: 230833, norm: 4.39532995223999
GEBURTSDATUM
idx: 229509, norm: 4.345174312591553
 pleaſure
idx: 210106, norm: 4.338034629821777
 purpoſe
idx: 65688, norm: 4.335690498352051
tagHelperRunner
idx: 107483, norm: 4.327722072601318
 CreateTagHelper
idx: 165182, norm: 4.314513206481934
 Jefus
idx: 84694, norm: 4.293426036834717
 autorytatywna
idx: 174788, norm: 4.2845025062561035
AndEndTag
idx: 195484, norm: 4.282397270202637
MigrationBuilder
idx: 230304, norm: 4.274115085601807
Personendaten
idx: 86434, n

In [ ]:
config = TRFConfig(
    target_modules=["down_proj"],
)
model = add_adapter_to_model(model, config)
print(model.state_dict().keys())

['layers.0.mlp.down_proj.down.default.weight', 'layers.0.mlp.down_proj.up.default.weight', 'layers.1.mlp.down_proj.down.default.weight', 'layers.1.mlp.down_proj.up.default.weight', 'layers.2.mlp.down_proj.down.default.weight', 'layers.2.mlp.down_proj.up.default.weight', 'layers.3.mlp.down_proj.down.default.weight', 'layers.3.mlp.down_proj.up.default.weight', 'layers.4.mlp.down_proj.down.default.weight', 'layers.4.mlp.down_proj.up.default.weight', 'layers.5.mlp.down_proj.down.default.weight', 'layers.5.mlp.down_proj.up.default.weight', 'layers.6.mlp.down_proj.down.default.weight', 'layers.6.mlp.down_proj.up.default.weight', 'layers.7.mlp.down_proj.down.default.weight', 'layers.7.mlp.down_proj.up.default.weight', 'layers.8.mlp.down_proj.down.default.weight', 'layers.8.mlp.down_proj.up.default.weight', 'layers.9.mlp.down_proj.down.default.weight', 'layers.9.mlp.down_proj.up.default.weight', 'layers.10.mlp.down_proj.down.default.weight', 'layers.10.mlp.down_proj.up.default.weight', 'layers

# FLUX.1

In [1]:
import numpy as np
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev")

/local2/kunkim/textboost-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 7/7 [00:04<00:00,  1.57it/s]


In [2]:
# print(pipe.text_encoder_2)
print(pipe.transformer)

FluxTransformer2DModel(
  (pos_embed): FluxPosEmbed()
  (time_text_embed): CombinedTimestepGuidanceTextProjEmbeddings(
    (time_proj): Timesteps()
    (timestep_embedder): TimestepEmbedding(
      (linear_1): Linear(in_features=256, out_features=3072, bias=True)
      (act): SiLU()
      (linear_2): Linear(in_features=3072, out_features=3072, bias=True)
    )
    (guidance_embedder): TimestepEmbedding(
      (linear_1): Linear(in_features=256, out_features=3072, bias=True)
      (act): SiLU()
      (linear_2): Linear(in_features=3072, out_features=3072, bias=True)
    )
    (text_embedder): PixArtAlphaTextProjection(
      (linear_1): Linear(in_features=768, out_features=3072, bias=True)
      (act_1): SiLU()
      (linear_2): Linear(in_features=3072, out_features=3072, bias=True)
    )
  )
  (context_embedder): Linear(in_features=4096, out_features=3072, bias=True)
  (x_embedder): Linear(in_features=64, out_features=3072, bias=True)
  (transformer_blocks): ModuleList(
    (0-18): 19 

In [6]:
embeddings = pipe.text_encoder.get_input_embeddings().weight.data.cpu().numpy()

# Remove zero-norm embeddings

embeddings = embeddings.astype(np.float32)[:-2]
norms = np.linalg.norm(embeddings, axis=1)
non_zero_norms = norms > 1e-6
embeddings = embeddings[non_zero_norms]
print(embeddings.shape)

# Min, mean, max norm
norms = np.linalg.norm(embeddings, axis=1)
print("Min norm:", norms.min())
print("Mean norm:", norms.mean())
print("Max norm:", norms.max())

(49199, 768)
Min norm: 0.0639129
Mean norm: 0.3869892
Max norm: 0.4772437


In [7]:
embeddings = pipe.text_encoder_2.get_input_embeddings().weight.data.cpu().numpy()

# Remove zero-norm embeddings

embeddings = embeddings.astype(np.float32)
norms = np.linalg.norm(embeddings, axis=1)
non_zero_norms = norms > 1e-6
embeddings = embeddings[non_zero_norms]
print(embeddings.shape)

# Min, mean, max norm
norms = np.linalg.norm(embeddings, axis=1)
print("Min norm:", norms.min())
print("Mean norm:", norms.mean())
print("Max norm:", norms.max())

(32128, 4096)
Min norm: 61.97627
Mean norm: 520.4404
Max norm: 778.97156
